# 深層強化学習

深層強化学習は、状態空間が大きい問題で価値関数や方策をニューラルネットで近似する方法の総称です。表形式では持てない連続状態や高次元観測を扱える点が実務上の利点です。

このノートでは、その中でも DQN 系の value-based deep RL に絞って扱います。policy-based や actor-critic は別系統の設計として、ここでは範囲外に置きます。


## 参考動画
- [対応動画 1](https://www.youtube.com/watch?v=2k5y3blCMbw)
@[youtube](https://www.youtube.com/watch?v=2k5y3blCMbw)
- [対応動画 2](https://www.youtube.com/watch?v=hE2A-IxHWc4)

## 参考リンク
- https://www.youtube.com/watch?v=2k5y3blCMbw
- https://www.youtube.com/watch?v=hE2A-IxHWc4


## 背景と目的

表形式Q学習は状態が増えるとメモリとサンプル効率の両面で限界があります。

小さなMLPでも、状態特徴から行動価値を近似できることを確認すると、DQN系の基本発想をつかめます。

ここでは NumPy だけで1隠れ層ネットを実装し、TD損失で更新します。


## 最初に解きたい疑問

1. 何が「深層」に置き換わっているのか。
2. one-hot 入力は、なぜ必要なのか。
3. `target = r + gamma * max(q_next)` は、表形式Q学習と何が同じで何が違うのか。
4. `done` が True のとき、なぜ target が変わるのか。
5. `target network` と `experience replay` を省くと、何が省かれているのか。


## 先に押さえる言葉

- `function approximation`: 表を持つ代わりに、関数で価値を近似することです。状態が多いときに使います。
- `one-hot encoding`: カテゴリを1本のベクトルで表す方法です。ここでは状態番号をニューラルネットに入れるために使います。
- `target value`: 学習で目標にするQ値です。`r` と次状態の推定値から作ります。
- `generalization`: ある状態で学んだ更新が、ほかの状態にも影響する性質です。ニューラルネットで起きます。


## 実行前の見取り図

1. `step=0/300/600/900...` の `Q(s0)` と `Q(s1)` がどう変わるか。
2. `target` と `err` が、学習中に更新方向を決めているか。
3. `forward(one_hot(0))` と `forward(one_hot(1))` の出力差が、学習で縮むか広がるか。


## つまずきやすい点

- MLP が表形式よりどこで強いのかを、状態数や一般化の観点で説明する補足がない。
- `done` と bootstrap をどう切り替えるかの説明が、コードだけでは弱い。


## コード 1: 更新式や補助関数を定義する

このセルでは 更新式や補助関数を定義する ための最小コードを動かします。 実行時は「`step=0/300/600/900...` の `Q(s0)` と `Q(s1)` がどう変わるか。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
import numpy as np
np.random.seed(0)

gamma = 0.95
lr = 0.03

# transition = (state, action, reward, next_state, done)
transitions = [
    (0, 1, 1.0, 1, False),
    (0, 0, 0.2, 0, False),
    (1, 0, 0.5, 0, False),
    (1, 1, 1.2, 1, True),
]

W1 = 0.3 * np.random.randn(2, 8)
b1 = np.zeros(8)
W2 = 0.3 * np.random.randn(8, 2)
b2 = np.zeros(2)


def one_hot(s):
    x = np.zeros(2)
    x[s] = 1.0
    return x


def forward(x):
    h_pre = x @ W1 + b1
    h = np.tanh(h_pre)
    q = h @ W2 + b2
    return h_pre, h, q


## コード 2: 入力データを作る

このセルでは 入力データを作る ための最小コードを動かします。 実行時は「`target` と `err` が、学習中に更新方向を決めているか。`terminal_updates` が増えたときに、終端遷移では bootstrap 項が切れているか。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
terminal_updates = 0

for step in range(1200):
    s, a, r, s_next, done = transitions[np.random.randint(len(transitions))]
    x = one_hot(s)
    xn = one_hot(s_next)

    h_pre, h, q = forward(x)
    _, _, q_next = forward(xn)

    if done:
        terminal_updates += 1
        target = r
    else:
        target = r + gamma * np.max(q_next)
    err = q[a] - target

    # backprop for selected action loss 0.5*(q[a]-target)^2
    dq = np.zeros_like(q)
    dq[a] = err

    dW2 = np.outer(h, dq)
    db2 = dq
    dh = W2 @ dq
    dh_pre = dh * (1.0 - np.tanh(h_pre) ** 2)
    dW1 = np.outer(x, dh_pre)
    db1 = dh_pre

    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

    if step % 300 == 0:
        q0 = forward(one_hot(0))[2]
        q1 = forward(one_hot(1))[2]
        print(f'step={step}: Q(s0)={np.round(q0,4)}, Q(s1)={np.round(q1,4)}, terminal_updates={terminal_updates}')


ネットワーク近似では、更新が他状態へ波及する一般化効果が得られます。この例では 1 本だけ `done=True` の終端遷移を混ぜてあり、そのときは `target = r` になって bootstrap 項が消えます。ここでの実装は最小構成で、DQNで一般的な target network と experience replay は省略しています。
